# Randomization Inference: Exactness Comes From the Assignment Mechanism

**Econometrics Notebook Library · v0.1.0**

## Intuition

Randomization inference treats the observed sample as fixed and randomness as coming from the treatment assignment mechanism. Under Fisher's sharp null, all missing potential outcomes are known, so the statistic can be recomputed over every assignment that the design could have produced.

For complete random assignment with $N_1$ treated units, the reference set is all $\binom{N}{N_1}$ assignments—not arbitrary permutations that violate the original design.

Review: [Athey & Imbens, The Econometrics of Randomized Experiments](https://arxiv.org/abs/1607.00698).

## Sharp null and constant additive effects

The sharp null $H_0:Y_i(1)-Y_i(0)=\tau_0$ lets us recover uniformity-trial outcomes

$$Y_i^{(0)}=Y_i^{obs}-\tau_0 Z_i.$$

For each allowed assignment $Z^*$, compute a test statistic such as the difference in means. The randomization $p$-value is the fraction of allowed assignments producing a statistic at least as extreme as observed.

This is finite-sample exact under the sharp null when the assignment mechanism is exactly reproduced.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

In [ ]:
from econnotes.core import simulate_randomized_experiment, randomization_inference_complete, difference_in_means

sim = simulate_randomized_experiment(n=70, n_treated=35, tau=0.65, seed=404)
obs = difference_in_means(sim["y"], sim["z"])
test_zero = randomization_inference_complete(sim["y"], sim["z"], tau_null=0.0, B=2500, seed=11)
test_true = randomization_inference_complete(sim["y"], sim["z"], tau_null=0.65, B=2500, seed=11)
{"observed_difference":obs, "p_sharp_null_0":test_zero["p_value"], "p_sharp_null_true_tau":test_true["p_value"]}

In [ ]:
fig, ax = plt.subplots()
ax.hist(test_zero["null_distribution"], bins=35, density=True, alpha=.7)
ax.axvline(test_zero["stat_obs"], linestyle="--", label="Observed under H0")
ax.set(xlabel="Difference in means under reassignment", ylabel="Density", title="Randomization distribution under the sharp null")
ax.legend();

## Inverting the test

Testing a grid of constant additive sharp nulls gives a randomization-based confidence set. This is conceptually different from attaching a robust standard error to the observed difference in means.

In [ ]:
grid = np.linspace(-0.2, 1.3, 21)
pvals = [randomization_inference_complete(sim["y"], sim["z"], tau_null=tau, B=500, seed=19)["p_value"] for tau in grid]
accepted = grid[np.array(pvals) >= .05]
(accepted.min(), accepted.max())

## Researcher failure checklist

- Reproduce the actual assignment mechanism: complete, blocked, paired, clustered, or rerandomized.
- Do not permute individuals when treatment was assigned by cluster.
- Distinguish Fisher's sharp null from a weak null of zero average treatment effect.
- Use a studentized statistic when theory for weak-null robustness requires it.
- Report Monte Carlo error when approximating a huge assignment space.